# Session 2.1 : Data Transformation

_Analytics Through Coding Autumn 2026_

---

Visualisation and statistical analysis are only useful if the data are in the form required to answer the question.

In practice, we often need to:

* keep only relevant observations;
* reorder observations;
* select or rename variables;
* create new variables;
* group observations;
* calculate summaries.

In this session we will use the `flights` dataset, which contains flights that departed from New York City in 2013.

Rather than learning pandas functions in isolation, we will start with an **analytical question** and then decide what transformation is needed.

---

## Starting out

As always, import the necessary libraries and dataset at the start of the notebook.

In [4]:
import pandas as pd
import numpy as np

flights = pd.read_csv("../Data/nycflights13_flights.csv", index_col=0)
flights.reset_index(drop=True, inplace=True)

flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00


Before transforming a dataset, remind yourself what one row represents and check the variables available.

In [5]:
print("Shape:", flights.shape)
print(flights.columns.tolist())

Shape: (202066, 19)
['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'time_hour']


## A small set of transformation tools

We will focus on a small number of pandas methods that can be combined to answer many analytical questions.

| pandas method | What it helps us do |
|---|---|
| `query()` | Keep observations that satisfy a condition |
| `sort_values()` | Reorder observations |
| `loc[]` | Select rows and/or columns |
| `rename()` | Rename variables |
| `assign()` | Create new variables |
| `groupby()` | Divide observations into groups |
| `agg()` | Calculate summaries |

The important skill is not memorising this table. It is recognising **which operation is required by the analytical question**.

## Question 1: Which flights departed on 16 August?

We only want observations where:

* `month == 8`
* `day == 16`

We can use `.query()` to filter observations.

In [32]:
flights_aug16 = flights.query("month == 8 and day == 16").copy()
flights_aug16.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
286,2013,8,16,1510.0,1455,15.0,1652.0,1701,-9.0,9E,4120,N8775A,JFK,CLE,73.0,425,14,55,2013-08-16 14:00:00
410,2013,8,16,1256.0,1255,1.0,1543.0,1545,-2.0,UA,1641,N19136,EWR,MCO,143.0,937,12,55,2013-08-16 12:00:00
707,2013,8,16,1959.0,2000,-1.0,2242.0,2310,-28.0,DL,2391,N916DL,JFK,TPA,146.0,1005,20,0,2013-08-16 20:00:00
1564,2013,8,16,619.0,620,-1.0,906.0,843,23.0,DL,1743,N6704Z,JFK,ATL,121.0,760,6,20,2013-08-16 06:00:00
1573,2013,8,16,2156.0,2159,-3.0,2258.0,2324,-26.0,UA,1116,N71411,EWR,BOS,42.0,200,21,59,2013-08-16 21:00:00


Notice that pandas returns a **new DataFrame**. The original `flights` DataFrame has not been changed.

In [33]:
print(flights_aug16.shape)

(608, 19)


Multiple arguments to `.query()` are combined with `“and”`: every expression must be true in order for a row to be included in the output. For some operations you may need other Boolean operations - `&` is “and”, `|` is “or”, and `!` is “not”

![GitHub Codespaces](Boolean_operators.png)

### Exercise 1

Find all flights that:

* departed from `JFK`;
* travelled to `LAX`; and
* had a departure delay greater than 60 minutes.

Keep the result in a DataFrame called `jfk_lax_delayed`.

How many flights meet all three conditions?

In [34]:
jfk_lax_delayed = flights.query("origin == 'JFK' and dest == 'LAX' and dep_delay > 60")
jfk_lax_delayed.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
689,2013,8,5,2206.0,1915,171.0,38.0,2230,128.0,AA,21,N327AA,JFK,LAX,302.0,2475,19,15,2013-08-05 19:00:00
832,2013,1,31,1011.0,900,71.0,1320.0,1227,53.0,DL,120,N624AG,JFK,LAX,338.0,2475,9,0,2013-01-31 09:00:00
1782,2013,6,1,1439.0,1300,99.0,1741.0,1555,106.0,VX,411,N627VA,JFK,LAX,334.0,2475,13,0,2013-06-01 13:00:00
2284,2013,5,17,1701.0,1545,76.0,1944.0,1910,34.0,AA,133,N338AA,JFK,LAX,316.0,2475,15,45,2013-05-17 15:00:00
2751,2013,6,28,2259.0,2150,69.0,143.0,100,43.0,AA,185,N335AA,JFK,LAX,320.0,2475,21,50,2013-06-28 21:00:00


## Question 2: Which flights experienced the largest departure delays?

Filtering determines **which observations we keep**.

Sorting determines **the order in which we inspect them**.

In [36]:
flights.sort_values("dep_delay", ascending=False)[["month", "day", "origin", "dest"]]


,month,day,origin,dest
114519,1,9,JFK,HNL
74463,1,10,EWR,ORD
176637,9,20,JFK,SFO
201196,3,17,LGA,MSP
53590,7,22,LGA,ATL
...,...,...,...,...
201790,2,8,LGA,RDU
201852,4,19,LGA,RDU
201856,1,17,EWR,RIC
201942,1,26,EWR,DFW


We can also sort using more than one variable.

For example, the following sorts chronologically by month and day.

In [38]:
flights.sort_values(by=["month", "day"], ascending=False)


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
1143,2013,12,31,657.0,659,-2.0,929.0,941,-12.0,DL,1547,N342NW,LGA,ATL,129.0,762,6,59,2013-12-31 06:00:00
1937,2013,12,31,651.0,700,-9.0,1014.0,959,15.0,DL,2285,N926DL,LGA,MCO,157.0,950,7,0,2013-12-31 07:00:00
2507,2013,12,31,1745.0,1745,0.0,2052.0,2100,-8.0,DL,2579,N339NB,LGA,FLL,169.0,1076,17,45,2013-12-31 17:00:00
2856,2013,12,31,1433.0,1430,3.0,1732.0,1739,-7.0,UA,1589,N66803,EWR,FLL,164.0,1065,14,30,2013-12-31 14:00:00
3898,2013,12,31,1952.0,1955,-3.0,2234.0,2236,-2.0,DL,2454,N372DA,JFK,DEN,248.0,1626,19,55,2013-12-31 19:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200352,2013,1,1,1959.0,2000,-1.0,2310.0,2307,3.0,UA,1233,N76514,EWR,IAH,232.0,1400,20,0,2013-01-01 20:00:00
200715,2013,1,1,743.0,749,-6.0,1043.0,1054,-11.0,B6,341,N624JB,JFK,SRQ,164.0,1041,7,49,2013-01-01 07:00:00
201488,2013,1,1,1320.0,1320,0.0,1625.0,1636,-11.0,UA,1425,N58101,EWR,LAX,334.0,2454,13,20,2013-01-01 13:00:00
201507,2013,1,1,1912.0,1915,-3.0,2200.0,2219,-19.0,DL,1485,N917DL,LGA,MCO,142.0,950,19,15,2013-01-01 19:00:00


### Exercise 2

Find the **10 flights with the longest arrival delays**.

Display only:

* `origin`
* `dest`
* `carrier`
* `arr_delay`

Sort the result from the largest delay to the smallest.

In [39]:
flights.sort_values("arr_delay", ascending=False)[["carrier", "day", "origin", "dest"]].head(10)    

,carrier,day,origin,dest
114519,HA,9,JFK,HNL
74463,MQ,10,EWR,ORD
176637,AA,20,JFK,SFO
201196,DL,17,LGA,MSP
53590,DL,22,LGA,ATL
150167,MQ,3,EWR,ORD
194528,DL,14,JFK,TPA
54279,AA,19,JFK,LAS
103118,MQ,1,JFK,BWI
151237,DL,19,EWR,SLC


## Question 3: Which variables do we actually need?

Real datasets often contain many more variables than are required for a particular question.

For an analysis of flight delays, we might only need a subset of columns.

In [6]:
delay_variables = flights.loc[
    :,
    ["month", 'day', 'carrier', 'dest', 'arr_delay', 'dep_delay']
]

We can also rename variables when a clearer name would make later code easier to read.

In [7]:
delay_variables.rename(
    columns={
        "dep-delay": "departure_delay",
        "arr_delay": "arrival_delay"
    },

).head()

,month,day,carrier,dest,arrival_delay,dep_delay
0,3,25,UA,RSW,19.0,24.0
1,4,26,DL,SFO,-37.0,-4.0
2,5,21,EV,DCA,16.0,11.0
3,7,18,EV,CLT,-22.0,-8.0
4,8,29,B6,BQN,0.0,-5.0


<div class="alert alert-warning">
<b>Note.</b>
Selecting or renaming columns does not improve the analysis by itself. Do it when it makes the dataset easier to understand or when only a smaller set of variables is needed for the question.
</div>

## Question 4: Did flights make up time while in the air?

Sometimes the variable we need does not exist in the original dataset.

We can create a new variable from existing variables using `.assign()`.

Define:

`gain = arrival delay - departure delay`

A negative value means the flight arrived with **less delay** than it had when it departed.

In [8]:
flight_with_gain = flights.assign(
    gain = flights["arr_delay"] - flights["dep_delay"]
)
flight_with_gain.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,gain
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,-5.0
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,-33.0
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,5.0
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,-14.0
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,5.0


We can create several new variables at the same time.

For example, approximate average speed in miles per hour can be calculated from `distance` and `air_time`.

In [11]:
flight_transsformed = flights.assign(
    gain = flights["arr_delay"] - flights["dep_delay"],
    speed = flights["distance"] / flights["air_time"] / 60)
flight_transsformed.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,gain,speed
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,...,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,-5.0,0.105325
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,...,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,-33.0,0.127893
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,...,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,5.0,0.085043
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,...,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,-14.0,0.114502
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,...,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,5.0,0.132660


### Exercise 3

Create a DataFrame called `flights_delay_change` containing a new variable:

`delay_change = arr_delay - dep_delay`

Then keep only flights where `delay_change <= -30`.

These are flights that reduced their delay by at least 30 minutes between departure and arrival.

Display the 10 flights with the largest reduction in delay.

In [12]:
flights_delay_change = (
    flights.assign(
        delay_change = flights["arr_delay"] - flights["dep_delay"]
    )
    .query("delay_change <= -30")
)

flights_delay_change.sort_values("delay_change").head(10)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,delay_change
62257,2013,6,13,1907.0,1512,235.0,2134.0,1928,126.0,EV,4377,N19554,EWR,JAX,126.0,820,15,12,2013-06-13 15:00:00,-109.0
10546,2013,2,26,1000.0,900,60.0,1513.0,1540,-27.0,HA,51,N382HA,JFK,HNL,584.0,4983,9,0,2013-02-26 09:00:00,-87.0
127816,2013,2,23,1226.0,900,206.0,1746.0,1540,126.0,HA,51,N389HA,JFK,HNL,599.0,4983,9,0,2013-02-23 09:00:00,-80.0
167806,2013,5,13,1917.0,1900,17.0,2149.0,2251,-62.0,DL,1465,N721TW,JFK,SFO,313.0,2586,19,0,2013-05-13 19:00:00,-79.0
102639,2013,2,27,924.0,900,24.0,1448.0,1540,-52.0,HA,51,N389HA,JFK,HNL,589.0,4983,9,0,2013-02-27 09:00:00,-76.0
5530,2013,7,14,1917.0,1829,48.0,2109.0,2135,-26.0,UA,673,N817UA,EWR,SNA,274.0,2434,18,29,2013-07-14 18:00:00,-74.0
55943,2013,5,2,1947.0,1949,-2.0,2209.0,2324,-75.0,UA,612,N851UA,EWR,LAX,300.0,2454,19,49,2013-05-02 19:00:00,-73.0
108369,2013,11,13,2024.0,2015,9.0,2251.0,2354,-63.0,DL,427,N188DN,JFK,LAX,311.0,2475,20,15,2013-11-13 20:00:00,-72.0
9831,2013,5,2,1949.0,1910,39.0,2208.0,2240,-32.0,AA,21,N339AA,JFK,LAX,301.0,2475,19,10,2013-05-02 19:00:00,-71.0
47345,2013,5,2,1926.0,1929,-3.0,2157.0,2310,-73.0,UA,1628,N24212,EWR,SFO,314.0,2565,19,29,2013-05-02 19:00:00,-70.0


## Question 5: What is a typical delay?

`.agg()` allows us to collapse many observations into summary statistics.

Without grouping, the summary describes the **entire dataset**.

In [13]:
flights.agg(
    mean_departure_delay = ("dep_delay", "mean"),
    median_departure_delay = ("dep_delay", "median"),
    mean_arrival_delay = ("arr_delay", "mean")
)

,dep_delay,arr_delay
mean_departure_delay,12.623232,NaN
median_departure_delay,-2.000000,NaN
mean_arrival_delay,NaN,6.944759


## Question 6: Does delay differ between airports or airlines?

Usually we want summaries **within groups**.

`groupby()` changes the unit of analysis.

Instead of one row representing one flight, the resulting table can have one row representing one airport, airline, month, destination, or another group.

In [28]:
delay_by_origin = (
    flights
    .groupby("origin")
    .agg(
        flights=("dep_delay", "size"),
        mean_dep_delay=("dep_delay", "mean"),
        mean_arr_delay=("arr_delay", "mean"),
    )
)

delay_by_origin.head()

,flights,mean_dep_delay,mean_arr_delay
origin,,,
EWR,72435,15.083630,9.136757
JFK,66846,12.109672,5.638486
LGA,62785,10.328891,5.817045


This is an important conceptual change:

> Before `groupby()`: one row = one flight  
> After `groupby()` + `agg()`: one row = one origin airport

Always know what **one row represents** after a transformation.

### Exercise 4

Calculate the following for each airline (`carrier`):

* `n_flights`: number of flights;
* `avg_dep_delay`: average departure delay;
* `avg_arr_delay`: average arrival delay.

Keep only airlines with at least 1,000 flights and sort them from the **lowest to highest average arrival delay**.

In [26]:
delay_by_carrier = (
    flights
    .groupby("carrier")
    .agg(
        n_flights=("dep_delay", "size"),
        avg_dep_delay=("dep_delay", "mean"),
        avg_arr_delay=("arr_delay", "mean"),
    )
    .query("n_flights >= 1000")
    .sort_values("avg_arr_delay")
)

delay_by_carrier.head()


,n_flights,avg_dep_delay,avg_arr_delay
carrier,,,
AA,19563,8.406411,0.383085
VX,3109,12.668714,1.578196
DL,28774,9.363554,1.848063
US,12384,3.866450,2.287291
UA,35205,11.869524,3.323555


## Combining transformations

Real analytical questions usually require more than one operation.

Method chaining lets us read the analysis as a sequence:

1. start with the data;
2. create or modify variables;
3. group observations;
4. calculate summaries;
5. filter the summaries;
6. order the result.

This mirrors the analytical workflow more closely than learning each method separately.

In [27]:
delay_by_carrier = (
    flights
    .groupby("carrier")
    .agg(
        n_flights=("dep_delay", "size"),
        avg_dep_delay=("dep_delay", "mean"),
        avg_arr_delay=("arr_delay", "mean"),
    )
    .query("n_flights >= 1000 & avg_arr_delay > 5")
    .reset_index()
    .sort_values("avg_arr_delay")
)

delay_by_carrier.head()

,carrier,n_flights,avg_dep_delay,avg_arr_delay
0,9E,11032,16.512983,7.021901
5,WN,7394,17.047298,9.123862
1,B6,32910,13.185801,9.764146
4,MQ,15806,10.901307,10.991669
2,EV,32519,19.976493,15.858449


### Final Exercise — Average speed by destination

Imagine you want to know how fast flights travel on average depending on destination.

Create a new variable:

`speed = distance / (air_time / 60)`

Then, for each destination (`dest`), calculate:

* `count`: number of flights;
* `avg_speed`: average speed in miles per hour.

Keep only destinations with **more than 20 flights** and sort them by `avg_speed` from fastest to slowest.

Display the 10 fastest destinations.

Try to write this as a single method chain.

In [31]:
destination_speed = (
    flights
    .assign(speed = flights["distance"] / flights["air_time"] / 60)
    .groupby("dest")
    .agg(
        count=("speed", "size"),
        avg_speed=("speed", "mean"),
    )
    .query("count > 20")
    .reset_index()
    .sort_values("avg_speed", ascending=False)
    .head(10)
)

destination_speed

,dest,count,avg_speed
11,BQN,547,0.135261
83,SJU,3521,0.134861
37,HNL,428,0.134311
69,PSE,229,0.133628
89,STT,289,0.132967
45,LAX,9687,0.125829
85,SMF,164,0.125418
76,SAN,1647,0.125410
46,LGB,390,0.124895
81,SFO,8027,0.124616


## Check the result

A transformation is not finished just because the code ran.

Before using the result, ask:

* Does one row now represent what I think it represents?
* Are the number of groups plausible?
* Are the summary values plausible?
* Did missing values affect the calculation?
* Did filtering happen before or after aggregation as intended?

A few simple checks can prevent incorrect conclusions.

## All Done!

In this session we used pandas transformations to answer analytical questions.

We practised:

* filtering observations with `query()`;
* sorting observations with `sort_values()`;
* selecting variables with `loc[]`;
* renaming variables;
* creating new variables with `assign()`;
* grouping observations with `groupby()`;
* calculating summaries with `agg()`;
* combining several operations using method chaining;
* checking that transformed results still make analytical sense.

The key idea is:

> **Start with the question, then decide what data transformation is required.**

We will now move on to the structure of datasets and the principles of **tidy data**.